- Взять 50-100 коров с реф панели, оставить только 24 хромосому, на ней какое-то кол-во SNP
- Создать реф панель, в которой нету взятых коров
- Конвертировать в FinalReport

In [464]:
!pwd


/app


In [465]:
!mkdir tests/preparation_data/temp


mkdir: cannot create directory ‘tests/preparation_data/temp’: File exists


In [466]:
!mkdir tests/data/raw


mkdir: cannot create directory ‘tests/data/raw’: File exists


In [467]:
OUTPUT_DIR = "tests/preparation_data/temp"

# choice = random.choices(range(9, 540), k=100)
# Не больше 530 ? !!!
SAMPLE_QUABTITY = (
    100  # k= Кол-во суммарное для тестовой реф панели и тестовой таргетной панели
)
REF_TARGET_DIVIDER = 0.3  # Процент определяющий размер таргетной панели


Подготовка example файлов, чтобы понимать, как они должны выглядеть, но без раскрытия частных данных

In [468]:
import datetime
import random

import pandas as pd

# Заголовок
header = {}
with open("data/test_data/thintergen_share_geno_VM2_1_FinalReport.txt", "r") as f:
    for i in range(0, 9):
        line = f.readline().split("\t")
        header[line[0]] = line[1] if len(line) > 1 else ""


final_report = pd.read_csv(
    "data/test_data/thintergen_share_geno_VM2_1_FinalReport.txt",
    sep="\t",
    skiprows=9,
)

for chr in [24, 25]:
    # Подготовка формата _FinalReport
    with open(f"{OUTPUT_DIR}/chr{chr}_FinalReport.txt", "w") as f:
        final_report_example: pd.DataFrame = final_report[final_report["Chr"] == chr]
        # final_report_24 = final_report_24.drop_duplicates(subset=["SNP Name"])
        final_report_example = final_report_example[
            (final_report_example["Sample ID"] == "HORUS007200114646")
            | (final_report_example["Sample ID"] == "HORUS003910309577")
        ]
        # Неправильные(придуманные) названия для образцов
        final_report_example["Sample ID"] = final_report_example["Sample ID"].map(
            lambda x: "SAMPLE01" if x == "HORUS007200114646" else "SAMPLE02"
        )
        # Случайные данные
        final_report_example["Allele1 - AB"] = final_report_example["Allele1 - AB"].map(
            lambda _: random.choices(["A", "B", "-"], weights=[40, 40, 10], k=1)[0]
        )
        final_report_example["Allele2 - AB"] = final_report_example["Allele2 - AB"].map(
            lambda _: random.choices(["A", "B", "-"], weights=[40, 40, 10], k=1)[0]
        )

        # Формирование заголовка с правильным числом SNP и образцов
        header["Processing Date"] = (
            f"{datetime.datetime.now().strftime('%m/%d/%Y %I:%M %p')}\n"
        )
        header["Num SNPs"] = f"{len(final_report_example['SNP Name'].unique())}\n"
        header["Total SNPs"] = f"{len(final_report_example['SNP Name'].unique())}\n"
        header["Num Samples"] = f"{len(final_report_example['Sample ID'].unique())}\n"
        header["Total Samples"] = f"{len(final_report_example['Sample ID'].unique())}\n"

        header_lst = []
        for i in zip(header.keys(), header.values()):
            if i[0].endswith("\n"):
                header_lst.extend(list(i))
            else:
                new_i = i[0] + "\t"
                header_lst.extend(list([new_i, i[1]]))
        # Запись в файл
        f.write("".join(header_lst))
        final_report_example.to_csv(f, index=False, sep="\t")
    # Подготовка формата _SampleMap
    with open(f"{OUTPUT_DIR}/chr{chr}_SampleMap.txt", "w") as f:
        f.write(
            "Index	Name	ID	Gender	Plate	Well	Group	Parent1	Parent2	Replicate	SentrixPosition\n"
        )
        for i in final_report_example["Sample ID"].unique():
            f.write(f"		{i}\n")
    # Подготовка формата _SNPMap
    with open(f"{OUTPUT_DIR}/chr{chr}_SNPMap.txt", "w") as f:
        f.write(
            "Index	Name	Chromosome	Position	GenTrain Score	SNP	ILMN_Strand	Customer Strand	NormID\n"
        )
        for i in final_report_example["SNP Name"].unique():
            chr_snp = final_report_example[final_report_example["SNP Name"] == i][
                "Position"
            ].iat[1]
            pos_snp = final_report_example[final_report_example["SNP Name"] == i][
                "Chr"
            ].iat[1]
            f.write(f"	{i}	{chr_snp}	{pos_snp}\n")


In [469]:
manifest = pd.read_csv(
    "data/manifest/BovineHD_B1.csv", sep=",", skiprows=7, skipfooter=23, engine="python"
)

manifest


,IlmnID,Name,IlmnStrand,SNP,AddressA_ID,AlleleA_ProbeSeq,AddressB_ID,AlleleB_ProbeSeq,GenomeBuild,Chr,...,Species,Source,SourceVersion,SourceStrand,SourceSeq,TopGenomicSeq,BeadSetID,Exp_Clusters,Intensity_Only,RefStrand
0,ARS-BFGL-BAC-10172-0_T_F_1511662585,ARS-BFGL-BAC-10172,TOP,[A/G],29744436,GGTCCCCAAAGTATGTGGTAGCACTTACTTATGTAAGTCATCACTC...,NaN,NaN,UMD_3.1,14,...,Bos taurus,UM3,0,TOP,CTCAGAAGTTGGTCCCCAAAGTATGTGGTAGCACTTACTTATGTAA...,CTCAGAAGTTGGTCCCCAAAGTATGTGGTAGCACTTACTTATGTAA...,260,3,0,-
1,ARS-BFGL-BAC-1020-0_B_R_1511662870,ARS-BFGL-BAC-1020,BOT,[T/C],43711500,GGATTTTCTTCAATGTTGTTTCAGTGGCATCCTTTATTTGACTGGA...,NaN,NaN,UMD_3.1,14,...,Bos taurus,UM3,0,TOP,GGATTGAACTCAGGTCTCCTGATTTCTCACTGAGCCATCTGGGAAG...,GGATTGAACTCAGGTCTCCTGATTTCTCACTGAGCCATCTGGGAAG...,282,3,0,+
2,ARS-BFGL-BAC-10245-0_B_F_1511658502,ARS-BFGL-BAC-10245,BOT,[T/C],19677417,CGCCTTCTGTTTTTCTTCTTCTCTCTTCCTGTTCTCTTTCTCTCTG...,NaN,NaN,UMD_3.1,14,...,Bos taurus,UM3,0,BOT,CCCACTTCCCCGCCTTCTGTTTTTCTTCTTCTCTCTTCCTGTTCTC...,TCCTTTCTAGGAGGACAGGCCTGAGTGGGGGCCTGGGAGGGGAAGA...,260,3,0,-
3,ARS-BFGL-BAC-10345-0_T_F_1511662909,ARS-BFGL-BAC-10345,TOP,[A/C],20791343,ACCATTCATTCTATTGCTTTGTGCTTCAAGTACTCCTGCAAATAAA...,NaN,NaN,UMD_3.1,14,...,Bos taurus,UM3,0,TOP,GGTATAGGGCACCATTCATTCTATTGCTTTGTGCTTCAAGTACTCC...,GGTATAGGGCACCATTCATTCTATTGCTTTGTGCTTCAAGTACTCC...,260,3,0,-
4,ARS-BFGL-BAC-10365-0_T_R_1511658265,ARS-BFGL-BAC-10365,TOP,[A/C],20611360,CAGACTCACAGCACTAATTTCTCATCTTCTCCCAGCCAATGTTTAT...,NaN,NaN,UMD_3.1,14,...,Bos taurus,UM3,0,BOT,CAAAAACTTCAAGTGTATTATCCCGTTTATATTACACTCCTGAGGA...,GGCTTGTTGGCAGACTCACAGCACTAATTTCTCATCTTCTCCCAGC...,260,3,0,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
777957,UA-IFASA-9744-0_T_F_1511694619,UA-IFASA-9744,TOP,[A/G],60644483,TTGGAAATGGGTGGGAGGGGTGGATTCTGACCTAACTTATAACCAG...,NaN,NaN,UMD_3.1,14,...,Bos taurus,UM3,0,TOP,AACTGATGAATTGGAAATGGGTGGGAGGGGTGGATTCTGACCTAAC...,AACTGATGAATTGGAAATGGGTGGGAGGGGTGGATTCTGACCTAAC...,260,3,0,-
777958,UA-IFASA-9774-0_B_F_1511694984,UA-IFASA-9774,BOT,[T/C],46701302,AACTGTGGACATAGAATGTCACCTGCTGTATCAGAAAACAGCCATC...,NaN,NaN,UMD_3.1,14,...,Bos taurus,UM3,0,BOT,CTCACTATTAAACTGTGGACATAGAATGTCACCTGCTGTATCAGAA...,CTCATTTTCTCCATCCTGAATCCATTGGGGGTGGAGAAGGGGACGG...,260,3,0,-
777959,UA-IFASA-9790-0_B_F_1511694684,UA-IFASA-9790,BOT,[T/C],72800491,GGAGGGCGAGGCTCCCGTTTCTTCATTTAGCAGATGTTCATAGAGC...,NaN,NaN,UMD_3.1,19,...,Bos taurus,UM3,0,BOT,ACCCATGATGGGAGGGCGAGGCTCCCGTTTCTTCATTTAGCAGATG...,TCAAGCAAGGCCCCACCCATCTGCTCTCTGCTGAGTCCTCAAGGTC...,260,3,0,+
777960,UA-IFASA-9812-0_T_R_1511695089,UA-IFASA-9812,TOP,[A/G],23731397,GGAACGTTTCTCTTACGTTTCTCTGCGTGCTTGAGAAACTTCAGAT...,NaN,NaN,UMD_3.1,29,...,Bos taurus,UM3,0,BOT,GTAAAAACAAACCTCCATAGCTGATAGGAATGGTCTCAACTTGCAG...,TCCCTTTGCTGGAACGTTTCTCTTACGTTTCTCTGCGTGCTTGAGA...,282,3,0,-


In [470]:
import pandas as pd

chr24_FinalReport = pd.read_csv(
    f"{OUTPUT_DIR}/chr24_FinalReport.txt",
    sep="\t",
    skiprows=9,
)
chr24_FinalReport


,SNP Name,Sample ID,Allele1 - AB,Allele2 - AB,Chr,Position
0,BovineHD2400000076,SAMPLE01,-,B,24,112601
1,BovineHD2400000069,SAMPLE01,B,-,24,163270
2,Hapmap54014-rs29018901,SAMPLE01,A,A,24,300821
3,BovineHD4100016339,SAMPLE01,B,A,24,335857
4,ARS-BFGL-NGS-103469,SAMPLE01,A,A,24,391651
...,...,...,...,...,...,...
3069,BovineHD2400018199,SAMPLE02,A,A,24,61954180
3070,BTB-01890193,SAMPLE02,A,A,24,61991351
3071,BovineHD2400018253,SAMPLE02,B,A,24,62039634
3072,ARS-BFGL-NGS-29191,SAMPLE02,B,B,24,62127707


In [471]:
# Сохранение SNP для извлечения из реф панели для таргетной панели
merge_manifest_and_fr = pd.merge(
    chr24_FinalReport, manifest, left_on="SNP Name", right_on="Name"
)
chr_24_snp = merge_manifest_and_fr.drop_duplicates(subset=["SNP Name"])[
    ["Chr_x", "Position"]
]
chr_24_snp.columns = ["CHROM", "POS"]
chr_24_snp.to_csv("data/ref_panel_test/chrom24_snp", index=False, sep="\t", header=None)
chr_24_snp


,CHROM,POS
0,24,112601
1,24,163270
2,24,300821
3,24,335857
4,24,391651
...,...,...
1457,24,61954180
1458,24,61991351
1459,24,62039634
1460,24,62127707


SNP - Chr - Position для 24 хромосомы

In [472]:
chr_24_snp = merge_manifest_and_fr.drop_duplicates(subset=["SNP Name"])
chr_24_snp = chr_24_snp[["SNP Name", "Chr_x", "Position"]]
chr_24_snp.columns = ["SNP Name", "Chr", "Position"]
chr_24_snp


,SNP Name,Chr,Position
0,BovineHD2400000076,24,112601
1,BovineHD2400000069,24,163270
2,Hapmap54014-rs29018901,24,300821
3,BovineHD4100016339,24,335857
4,ARS-BFGL-NGS-103469,24,391651
...,...,...,...
1457,BovineHD2400018199,24,61954180
1458,BTB-01890193,24,61991351
1459,BovineHD2400018253,24,62039634
1460,ARS-BFGL-NGS-29191,24,62127707


In [473]:
# Сохранение SNP для извлечения из реф панели для таргетной панели
chr25_FinalReport = pd.read_csv(
    f"{OUTPUT_DIR}/chr25_FinalReport.txt",
    sep="\t",
    skiprows=9,
)
merge_manifest_and_fr = pd.merge(
    chr25_FinalReport, manifest, left_on="SNP Name", right_on="Name"
)
chr_25_snp = merge_manifest_and_fr.drop_duplicates(subset=["SNP Name"])[
    ["Chr_x", "Position"]
]
chr_25_snp.columns = ["CHROM", "POS"]
chr_25_snp.to_csv("data/ref_panel_test/chrom25_snp", index=False, sep="\t", header=None)
chr_25_snp


,CHROM,POS
0,25,87242
1,25,109981
2,25,143925
3,25,150688
4,25,179006
...,...,...
1167,25,42167568
1168,25,42209916
1169,25,42218897
1170,25,42244411


SNP - Chr - Position для 24 хромосомы

In [474]:
chr_25_snp = merge_manifest_and_fr.drop_duplicates(subset=["SNP Name"])
chr_25_snp = chr_25_snp[["SNP Name", "Chr_x", "Position"]]
chr_25_snp.columns = ["SNP Name", "Chr", "Position"]
chr_25_snp


,SNP Name,Chr,Position
0,BovineHD2500000011,25,87242
1,ARS-BFGL-NGS-28648,25,109981
2,ARS-BFGL-NGS-67788,25,143925
3,BovineHD4100016786,25,150688
4,ARS-BFGL-BAC-37937,25,179006
...,...,...,...
1167,ARS-BFGL-NGS-27183,25,42167568
1168,ARS-BFGL-NGS-35357,25,42209916
1169,BovineHD4100017605,25,42218897
1170,BovineHD2500011969,25,42244411


Выбрать только Голштинскую породу из реф панели

In [475]:
df = pd.read_csv(
    "data/ref_panel/breed.csv",
)
df[df["Breed"] == "Holstein"][["SampleID"]].to_csv(
    "data/ref_panel_test/holstein_in_ref.txt", index=False, header=False
)


In [476]:
%%bash
OUT_FILE="data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz"

if [ ! -f "$OUT_FILE" ]; then
    echo "Файл $OUT_FILE не найден. Выполняем команды..."
    /opt/tools/bin/bcftools index data/ref_panel/Chr24_phased_snp.vcf.gz
    /opt/tools/bin/bcftools view -r 24 -S data/ref_panel_test/holstein_in_ref.txt \
        data/ref_panel/Chr24_phased_snp.vcf.gz -Oz -o "$OUT_FILE"
else
    echo "Файл $OUT_FILE уже существует. Пропускаем."
fi


Файл data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz уже существует. Пропускаем.


In [477]:
%%bash
OUT_FILE="data/ref_panel_test/Chr25_phased_snp_holstein_only.vcf.gz"

if [ ! -f "$OUT_FILE" ]; then
    echo "Файл $OUT_FILE не найден. Выполняем команды..."
    /opt/tools/bin/bcftools index data/ref_panel/Chr25_phased_snp.vcf.gz
    /opt/tools/bin/bcftools view -r 25 -S data/ref_panel_test/holstein_in_ref.txt \
        data/ref_panel/Chr25_phased_snp.vcf.gz -Oz -o "$OUT_FILE"
else
    echo "Файл $OUT_FILE уже существует. Пропускаем."
fi


Файл data/ref_panel_test/Chr25_phased_snp_holstein_only.vcf.gz уже существует. Пропускаем.


Выбор кол-ва животных голтинов в тестовой реф и таргетной панелях

In [478]:
choice = random.choices(range(9, 540), k=SAMPLE_QUABTITY)

divider = round(len(choice) * REF_TARGET_DIVIDER)
target_panel = choice[0:divider]

ref_panel = choice[divider:]

print(len(target_panel), target_panel)
print(len(ref_panel), ref_panel)


30 [447, 304, 75, 261, 35, 286, 126, 274, 40, 508, 150, 339, 41, 458, 216, 103, 362, 411, 80, 165, 364, 489, 510, 98, 422, 412, 514, 254, 405, 48]
70 [398, 143, 446, 224, 37, 255, 200, 446, 55, 344, 69, 232, 321, 467, 265, 323, 146, 504, 78, 342, 91, 500, 31, 337, 160, 59, 18, 92, 103, 89, 380, 233, 233, 153, 325, 119, 469, 200, 66, 459, 388, 141, 15, 322, 442, 258, 324, 77, 533, 200, 120, 345, 42, 135, 390, 114, 326, 367, 471, 241, 276, 495, 416, 239, 419, 293, 457, 307, 233, 12]


In [479]:
# 2985 - колонок
# 2985 - 9(Колонки vcf) = 2976 образцов
# 1580111 строк
df = pd.read_csv(
    "data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz",
    sep=r"\s+",
    skiprows=15,
    compression="gzip",
    nrows=1,
    usecols=[*range(0, 9), *target_panel],
    engine="python",
)
print(list(df.columns)[9:])
with open("data/ref_panel_test/target_sample", "w") as f:
    f.write("\n".join(list(df.columns)[9:]))
df


['19830', '20830', '20831', 'SAMEA3706814', 'SAMEA19325668', 'SAMEA5159881', 'SAMEA4644731', 'SAMEA4644748', 'SAMEA19876918', 'SAMEA4780264', 'SAMEA4780269', 'SAMEA4780319', 'SAMEA7690222', 'SAMEA7690256', 'SAMEA7690253', 'SAMEA3390144', 'SAMEA3390147', 'SAMN02843175', 'SAMN02225737', 'SAMN04576214', 'SAMN05942194', 'SAMN19491817', 'SAMN08612422', 'SAMN19491805', 'SAMN19491809', 'SAMN08612397', 'SAMN19491821', 'SAMN08612413', 'SAMN08612382', 'SAMN08612384']


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,19830,...,SAMN05942194,SAMN19491817,SAMN08612422,SAMN19491805,SAMN19491809,SAMN08612397,SAMN19491821,SAMN08612413,SAMN08612382,SAMN08612384
0,24,997,24_997_A_G,A,G,.,PASS,AN=1098;AC=2,GT,0|0,...,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0


In [480]:
df = pd.read_csv(
    "data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz",
    sep=r"\s+",
    skiprows=15,
    compression="gzip",
    nrows=1,
    usecols=[*range(0, 9), *ref_panel],
    engine="python",
)
print(list(df.columns)[9:])
with open("data/ref_panel_test/ref_sample", "w") as f:
    f.write("\n".join(list(df.columns)[9:]))


['15864', '18820', '18834', '19807', '20826', '20832', 'SAMEA5159854', 'SAMEA5159856', 'SAMEA5159857', 'SAMEA5415503', 'SAMEA32995168', 'SAMEA5160021', 'SAMEA4644726', 'SAMEA5415492', 'SAMEA4644732', 'SAMEA4644748', 'SAMEA19310668', 'SAMEA19848418', 'SAMEA33004168', 'SAMEA19867168', 'SAMEA4780320', 'SAMEA4780260', 'SAMEA4780283', 'SAMEA4780271', 'SAMEA4780265', 'SAMEA4780244', 'SAMEA7690258', 'SAMEA7690249', 'SAMEA7690201', 'SAMEA7690247', 'SAMEA7690250', 'SAMEA7690252', 'SAMEA7690223', 'SAMEA7015112', 'SAMEA7690262', 'SAMEA3390146', 'SAMEA3390162', 'SAMN02843161', 'SAMN02843128', 'SAMN02843156', 'SAMN02843162', 'SAMN02843129', 'SAMN02843164', 'SAMN02843177', 'SAMN02843169', 'SAMN02841109', 'SAMN02843154', 'SAMN10940614', 'SAMN10940590', 'SAMN10940598', 'SAMN10940588', 'SAMN10940604', 'SAMN19491801', 'SAMN08612443', 'SAMN08612399', 'SAMN08612409', 'SAMN08612386', 'SAMN08612380', 'SAMN08612403', 'SAMN19491814', 'SAMN19491800', 'SAMN08612442', 'SAMN08612410', 'SAMN08612401', 'SAMN0951962

## Подготовка тестовой референсной панели

In [481]:
%%bash

OUT_FILE="data/ref_panel_test/Chr24_ref.vcf.gz"

if [ ! -f "$OUT_FILE" ]; then
    echo "Файл $OUT_FILE не найден. Выполняем команды..."
    /opt/tools/bin/bcftools index data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz
    /opt/tools/bin/bcftools view -r 24 -S data/ref_panel_test/ref_sample \
        data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz \
        -Oz -o "$OUT_FILE"
else
    echo "Файл $OUT_FILE уже существует. Пропускаем."
fi


Файл data/ref_panel_test/Chr24_ref.vcf.gz уже существует. Пропускаем.


In [482]:
%%bash
OUT_FILE="data/ref_panel_test/Chr25_ref.vcf.gz"

if [ ! -f "$OUT_FILE" ]; then
    echo "Файл $OUT_FILE не найден. Выполняем команды..."
    /opt/tools/bin/bcftools index data/ref_panel_test/Chr25_phased_snp_holstein_only.vcf.gz
    /opt/tools/bin/bcftools view -r 25 -S data/ref_panel_test/ref_sample \
        data/ref_panel_test/Chr25_phased_snp_holstein_only.vcf.gz \
        -Oz -o "$OUT_FILE"
else
    echo "Файл $OUT_FILE уже существует. Пропускаем."
fi


Файл data/ref_panel_test/Chr25_ref.vcf.gz уже существует. Пропускаем.


## Подготовка тестовой таргетной панели

In [483]:
%%bash
OUT_FILE="data/ref_panel_test/Chr24_target.vcf.gz"

if [ ! -f "$OUT_FILE" ]; then
    echo "Файл $OUT_FILE не найден. Выполняем команды..."
    /opt/tools/bin/bcftools index data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz
    /opt/tools/bin/bcftools view -r 24 -S data/ref_panel_test/target_sample \
        -R data/ref_panel_test/chrom24_snp \
        data/ref_panel_test/Chr24_phased_snp_holstein_only.vcf.gz \
        -Oz -o "$OUT_FILE"
else
    echo "Файл $OUT_FILE уже существует. Пропускаем."
fi


Файл data/ref_panel_test/Chr24_target.vcf.gz уже существует. Пропускаем.


In [484]:
%%bash
OUT_FILE="data/ref_panel_test/Chr25_target.vcf.gz"

if [ ! -f "$OUT_FILE" ]; then
    echo "Файл $OUT_FILE не найден. Выполняем команды..."
    /opt/tools/bin/bcftools index data/ref_panel_test/Chr25_phased_snp_holstein_only.vcf.gz
    /opt/tools/bin/bcftools view -r 25 -S data/ref_panel_test/target_sample \
        -R data/ref_panel_test/chrom25_snp \
        data/ref_panel_test/Chr25_phased_snp_holstein_only.vcf.gz \
        -Oz -o "$OUT_FILE"
else
    echo "Файл $OUT_FILE уже существует. Пропускаем."
fi


Файл data/ref_panel_test/Chr25_target.vcf.gz уже существует. Пропускаем.


Не фазированный тип:


In [485]:
import pandas as pd


In [486]:
def not_phased(gt):
    gt = sorted(gt.split("|"))
    return "/".join(gt)


target_panel_chr24 = pd.read_csv(
    "data/ref_panel_test/Chr24_target.vcf.gz",
    sep=r"\s+",
    skiprows=16,
    compression="gzip",
    engine="python",
)


sample_columns = target_panel_chr24.columns[9:]

for i in sample_columns:
    target_panel_chr24[i] = target_panel_chr24[i].map(not_phased)

target_panel_chr24


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,18828,...,SAMN02843172,SAMN02225739,SAMN10940591,SAMN10940583,SAMN08612411,SAMN08612415,SAMN08612382,SAMN08612432,SAMN08612412,SAMN09519608
0,24,112601,24_112601_T_C,T,C,.,PASS,AN=54;AC=21,GT,0/0,...,0/0,0/1,0/1,1/1,0/1,0/0,0/0,0/0,0/1,0/1
1,24,163270,24_163270_C_T,C,T,.,PASS,AN=54;AC=25,GT,0/1,...,1/1,0/1,0/0,1/1,0/1,0/0,0/1,0/0,0/1,0/1
2,24,300821,24_300821_T_C,T,C,.,PASS,AN=54;AC=35,GT,0/1,...,1/1,1/1,0/1,1/1,1/1,0/1,0/1,0/0,1/1,0/1
3,24,335857,24_335857_A_G,A,G,.,PASS,AN=54;AC=13,GT,0/0,...,0/0,1/1,0/0,0/1,0/0,0/0,0/0,0/0,0/0,0/1
4,24,391651,24_391651_A_G,A,G,.,PASS,AN=54;AC=31,GT,1/1,...,0/0,0/1,0/1,1/1,0/1,0/1,0/0,0/0,1/1,0/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,24,61954180,24_61954180_A_G,A,G,.,PASS,AN=54;AC=23,GT,1/1,...,1/1,0/1,0/1,0/1,0/1,0/1,1/1,0/0,0/1,0/0
1378,24,61991351,24_61991351_G_A,G,A,.,PASS,AN=54;AC=54,GT,1/1,...,1/1,1/1,1/1,1/1,1/1,1/1,1/1,1/1,1/1,1/1
1379,24,62039634,24_62039634_T_G,T,G,.,PASS,AN=54;AC=24,GT,0/0,...,0/0,0/0,0/1,1/1,0/1,0/0,0/0,0/1,0/0,1/1
1380,24,62127707,24_62127707_G_A,G,A,.,PASS,AN=54;AC=18,GT,1/1,...,1/1,0/1,0/1,0/0,0/1,0/1,0/0,0/0,0/1,0/0


In [487]:
def not_phased(gt):
    gt = sorted(gt.split("|"))
    return "/".join(gt)


target_panel_chr25 = pd.read_csv(
    "data/ref_panel_test/Chr25_target.vcf.gz",
    sep=r"\s+",
    skiprows=16,
    compression="gzip",
    engine="python",
)


sample_columns = target_panel_chr25.columns[9:]

for i in sample_columns:
    target_panel_chr25[i] = target_panel_chr25[i].map(not_phased)

target_panel_chr25


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,18828,...,SAMN02843172,SAMN02225739,SAMN10940591,SAMN10940583,SAMN08612411,SAMN08612415,SAMN08612382,SAMN08612432,SAMN08612412,SAMN09519608
0,25,87242,25_87242_A_G,A,G,.,PASS,AN=54;AC=47,GT,1/1,...,0/1,1/1,0/1,0/1,1/1,1/1,1/1,1/1,1/1,1/1
1,25,109981,25_109981_G_A,G,A,.,PASS,AN=54;AC=0,GT,0/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
2,25,143925,25_143925_G_A,G,A,.,PASS,AN=54;AC=40,GT,1/1,...,0/1,1/1,0/1,0/1,1/1,1/1,0/1,1/1,1/1,1/1
3,25,150688,25_150688_G_A,G,A,.,PASS,AN=54;AC=32,GT,1/1,...,0/0,1/1,0/1,0/1,1/1,1/1,0/0,1/1,1/1,1/1
4,25,179006,25_179006_C_T,C,T,.,PASS,AN=54;AC=51,GT,1/1,...,1/1,1/1,1/1,1/1,1/1,1/1,1/1,1/1,1/1,1/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1099,25,42167568,25_42167568_T_C,T,C,.,PASS,AN=54;AC=15,GT,0/1,...,0/1,0/1,0/0,0/0,0/1,0/0,1/1,0/1,0/0,0/0
1100,25,42209916,25_42209916_T_C,T,C,.,PASS,AN=54;AC=27,GT,0/0,...,0/0,0/0,1/1,0/0,0/0,1/1,0/0,1/1,0/0,1/1
1101,25,42218897,25_42218897_G_A,G,A,.,PASS,AN=54;AC=17,GT,0/0,...,0/0,0/0,1/1,0/1,0/1,0/0,0/0,0/0,0/0,0/0
1102,25,42244411,25_42244411_C_T,C,T,.,PASS,AN=54;AC=39,GT,0/0,...,0/1,1/1,1/1,1/1,1/1,0/1,1/1,0/1,1/1,0/1


## Конвертация в FinalReport Format

In [488]:
chr_24_snp_test = pd.merge(
    chr_24_snp, manifest, left_on="SNP Name", right_on="Name", suffixes=(None, "y")
)
chr_24_snp = chr_24_snp_test[
    ["SNP Name", "Chr", "Position", "SNP", "IlmnStrand", "RefStrand"]
]
chr_24_snp


,SNP Name,Chr,Position,SNP,IlmnStrand,RefStrand
0,BovineHD2400000076,24,112601,[T/C],BOT,+
1,BovineHD2400000069,24,163270,[T/C],BOT,+
2,Hapmap54014-rs29018901,24,300821,[T/C],BOT,+
3,BovineHD4100016339,24,335857,[T/C],BOT,-
4,ARS-BFGL-NGS-103469,24,391651,[A/G],TOP,+
...,...,...,...,...,...,...
1457,BovineHD2400018199,24,61954180,[T/C],BOT,-
1458,BTB-01890193,24,61991351,[A/G],TOP,+
1459,BovineHD2400018253,24,62039634,[A/C],TOP,-
1460,ARS-BFGL-NGS-29191,24,62127707,[A/G],TOP,+


In [489]:
chr_25_snp_test = pd.merge(
    chr_25_snp, manifest, left_on="SNP Name", right_on="Name", suffixes=(None, "y")
)
chr_25_snp = chr_25_snp_test[
    ["SNP Name", "Chr", "Position", "SNP", "IlmnStrand", "RefStrand"]
]
chr_25_snp


,SNP Name,Chr,Position,SNP,IlmnStrand,RefStrand
0,BovineHD2500000011,25,87242,[A/G],TOP,+
1,ARS-BFGL-NGS-28648,25,109981,[T/C],BOT,-
2,ARS-BFGL-NGS-67788,25,143925,[T/C],BOT,-
3,BovineHD4100016786,25,150688,[A/G],TOP,+
4,ARS-BFGL-BAC-37937,25,179006,[A/G],TOP,-
...,...,...,...,...,...,...
1167,ARS-BFGL-NGS-27183,25,42167568,[T/C],BOT,+
1168,ARS-BFGL-NGS-35357,25,42209916,[T/C],BOT,+
1169,BovineHD4100017605,25,42218897,[A/G],TOP,+
1170,BovineHD2500011969,25,42244411,[A/G],TOP,-


In [490]:
snp_info_all_chr = pd.merge(chr_24_snp, chr_25_snp, how="outer")
snp_info_all_chr


,SNP Name,Chr,Position,SNP,IlmnStrand,RefStrand
0,ARS-BFGL-BAC-27530,25,10999339,[A/G],TOP,+
1,ARS-BFGL-BAC-2799,24,9687593,[A/G],TOP,+
2,ARS-BFGL-BAC-2814,24,16947207,[A/G],TOP,-
3,ARS-BFGL-BAC-28286,24,37374025,[T/G],BOT,+
4,ARS-BFGL-BAC-28633,24,23230058,[T/C],BOT,-
...,...,...,...,...,...,...
2629,UA-IFASA-7169,25,16538323,[A/G],TOP,-
2630,UA-IFASA-7925,24,438309,[T/C],BOT,+
2631,UA-IFASA-8558,24,37166678,[T/C],BOT,+
2632,UA-IFASA-8571,24,33353647,[T/C],BOT,-


In [491]:
target_panel_all_chr = pd.merge(target_panel_chr24, target_panel_chr25, how="outer")
target_panel_all_chr


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,18828,...,SAMN02843172,SAMN02225739,SAMN10940591,SAMN10940583,SAMN08612411,SAMN08612415,SAMN08612382,SAMN08612432,SAMN08612412,SAMN09519608
0,24,112601,24_112601_T_C,T,C,.,PASS,AN=54;AC=21,GT,0/0,...,0/0,0/1,0/1,1/1,0/1,0/0,0/0,0/0,0/1,0/1
1,24,163270,24_163270_C_T,C,T,.,PASS,AN=54;AC=25,GT,0/1,...,1/1,0/1,0/0,1/1,0/1,0/0,0/1,0/0,0/1,0/1
2,24,300821,24_300821_T_C,T,C,.,PASS,AN=54;AC=35,GT,0/1,...,1/1,1/1,0/1,1/1,1/1,0/1,0/1,0/0,1/1,0/1
3,24,335857,24_335857_A_G,A,G,.,PASS,AN=54;AC=13,GT,0/0,...,0/0,1/1,0/0,0/1,0/0,0/0,0/0,0/0,0/0,0/1
4,24,391651,24_391651_A_G,A,G,.,PASS,AN=54;AC=31,GT,1/1,...,0/0,0/1,0/1,1/1,0/1,0/1,0/0,0/0,1/1,0/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2481,25,42167568,25_42167568_T_C,T,C,.,PASS,AN=54;AC=15,GT,0/1,...,0/1,0/1,0/0,0/0,0/1,0/0,1/1,0/1,0/0,0/0
2482,25,42209916,25_42209916_T_C,T,C,.,PASS,AN=54;AC=27,GT,0/0,...,0/0,0/0,1/1,0/0,0/0,1/1,0/0,1/1,0/0,1/1
2483,25,42218897,25_42218897_G_A,G,A,.,PASS,AN=54;AC=17,GT,0/0,...,0/0,0/0,1/1,0/1,0/1,0/0,0/0,0/0,0/0,0/0
2484,25,42244411,25_42244411_C_T,C,T,.,PASS,AN=54;AC=39,GT,0/0,...,0/1,1/1,1/1,1/1,1/1,0/1,1/1,0/1,1/1,0/1


In [492]:
def convert_to_fr(gt, ref, alt, row):
    try:
        complementarity = {"A": "T", "T": "A", "C": "G", "G": "C"}
        SNP_A = row.SNP.strip("[]").split("/")[0]
        SNP_B = row.SNP.strip("[]").split("/")[1]
        IlmnStrand = row.IlmnStrand
        RefStrand = row.RefStrand
        REF = ref
        ALT = alt
        alphabetical_order = ["A", "C", "G", "T"]  # Правильный ?

        # if row.Allele1_AB != "-" and row.Allele2_AB != "-":
        if f"{SNP_A}/{SNP_B}" not in ["A/T", "C/G", "T/A", "G/C"]:
            if SNP_A == "A" or SNP_A == "T":
                allele_A = SNP_A
            else:
                allele_B = SNP_A
            if SNP_B == "C" or SNP_B == "G":
                allele_B = SNP_B
            else:
                allele_A = SNP_B
        elif f"{SNP_A}/{SNP_B}" in ["A/T", "C/G", "T/A", "G/C"]:
            num_SNP_A = alphabetical_order.index(SNP_A)  # 0, 1, 2, 3
            num_SNP_B = alphabetical_order.index(SNP_B)  # 0, 1, 2, 3

            if IlmnStrand == "TOP":
                allele_A = SNP_A if num_SNP_A < num_SNP_B else SNP_B
                allele_B = SNP_A if allele_A != SNP_A else SNP_B
            elif IlmnStrand == "BOT":
                allele_A = SNP_A if num_SNP_A > num_SNP_B else SNP_B
                allele_B = SNP_A if allele_A != SNP_A else SNP_B
        if RefStrand == "-":
            allele_A = complementarity[allele_A]
            allele_B = complementarity[allele_B]
        # SNP_normal = {"A": allele_A, "B": allele_B}
        SNP_normal_rev = {allele_A: "A", allele_B: "B"}

        # print(SNP_normal)
        # allele1 = 0 if SNP_normal[row.Allele1_AB] == REF else 1
        # allele2 = 0 if SNP_normal[row.Allele2_AB] == REF else 1
        gt = gt.split("/")
        # print(gt)
        gt = [REF if i == "0" else ALT for i in gt]
        # print(gt)
        result = {
            "Allele1_AB": SNP_normal_rev[gt[0]],
            "Allele2_AB": SNP_normal_rev[gt[1]],
        }
        return result
    except KeyError:
        result = {
            "Allele1_AB": "-",
            "Allele2_AB": "-",
        }
        return result


convert_to_fr(
    "0/1",
    "A",
    "G",
    chr_25_snp[chr_25_snp["Position"] == 87242].iloc[0],
)


{'Allele1_AB': 'A', 'Allele2_AB': 'B'}

In [493]:
sample_columns


Index(['18828', '19830', 'SAMEA5159859', 'SAMEA4644736', 'SAMEA33000418',
       'SAMEA4644767', 'SAMEA32984668', 'SAMEA4780279', 'SAMEA4780314',
       'SAMEA4780255', 'SAMEA4780318', 'SAMEA4780245', 'SAMEA7690246',
       'SAMEA3390141', 'SAMN02843163', 'SAMN02843167', 'SAMN02843162',
       'SAMN02843172', 'SAMN02225739', 'SAMN10940591', 'SAMN10940583',
       'SAMN08612411', 'SAMN08612415', 'SAMN08612382', 'SAMN08612432',
       'SAMN08612412', 'SAMN09519608'],
      dtype='object')

In [494]:
# SNP Name	Sample ID	Allele1 - AB	Allele2 - AB	Chr	Position
# 0	BovineHD2400000076	SAMPLE0	A	A	24	112601

# chr_25_snp SNP Name	Chr_x	Position


# sample_columns - чтобы поделить на 2 файла, а значит два архива

for set_name, sample_columns_set in [
    ("set01", sample_columns[: len(sample_columns) // 2]),
    ("set02", sample_columns[len(sample_columns) // 2 :]),
]:
    result = []
    for index, row in target_panel_all_chr.iterrows():
        for i in sample_columns_set:
            snp_info_row = snp_info_all_chr[
                snp_info_all_chr["Position"] == row["POS"]
            ].iloc[0]
            SNP_name = snp_info_row["SNP Name"]
            allele_AB = convert_to_fr(
                row[i],
                row["REF"],
                row["ALT"],
                snp_info_row,
            )
            result.append(
                {
                    "SNP Name": SNP_name,
                    "Sample ID": i,
                    "Allele1 - AB": allele_AB["Allele1_AB"],
                    "Allele2 - AB": allele_AB["Allele2_AB"],
                    "Chr": row["#CHROM"],
                    "Position": row["POS"],
                }
            )
    df = pd.DataFrame(result)
    with open(
        f"{OUTPUT_DIR}/target_panel_{set_name}_FinalReport.txt",
        "w",
    ) as f:
        header["Processing Date"] = (
            f"{datetime.datetime.now().strftime('%m/%d/%Y %I:%M %p')}\n"
        )
        header["Num SNPs"] = f"{len(df['SNP Name'].unique())}\n"
        header["Total SNPs"] = f"{len(df['SNP Name'].unique())}\n"
        header["Num Samples"] = f"{len(df['Sample ID'].unique())}\n"
        header["Total Samples"] = f"{len(df['Sample ID'].unique())}\n"

        header_lst = []
        for i in zip(header.keys(), header.values()):
            if i[0].endswith("\n"):
                header_lst.extend(list(i))
            else:
                new_i = i[0] + "\t"
                header_lst.extend(list([new_i, i[1]]))
        # Запись в файл
        f.write("".join(header_lst))
        df.sort_values("Sample ID").to_csv(
            f,
            sep="\t",
            index=False,
        )
    with open(
        f"{OUTPUT_DIR}/target_panel_{set_name}_SampleMap.txt",
        "w",
    ) as f:
        f.write(
            "Index	Name	ID	Gender	Plate	Well	Group	Parent1	Parent2	Replicate	SentrixPosition\n"
        )
        for i in df["Sample ID"].unique():
            f.write(f"		{i}\n")
    # Подготовка формата _SNPMap
    with open(
        f"{OUTPUT_DIR}/target_panel_{set_name}_SNPMap.txt",
        "w",
    ) as f:
        f.write(
            "Index	Name	Chromosome	Position	GenTrain Score	SNP	ILMN_Strand	Customer Strand	NormID\n"
        )
        for i in df["SNP Name"].unique():
            chr_snp = df[df["SNP Name"] == i]["Position"].iat[1]
            pos_snp = df[df["SNP Name"] == i]["Chr"].iat[1]
            f.write(f"	{i}	{chr_snp}	{pos_snp}\n")


In [495]:
%%bash
# test_data.zip
#  ├── sample1_FinalReport.txt
#  ├── subdir/
#  │   └── sample2_FinalReport.txt
#  └── __MACOSX/ (игнорируется)
cd /app/tests/preparation_data/temp

# Удаляем старый общий архив, если есть
rm -f test_data.zip

# set01
mkdir -p test_arch_set01/__MACOSX test_arch_set01/data/set01 test_arch_set01/maps
cp target_panel_set01_FinalReport.txt test_arch_set01/data/set01/
cp target_panel_set01_SNPMap.txt test_arch_set01/data/set01/
cp target_panel_set01_SampleMap.txt test_arch_set01/maps/
touch test_arch_set01/__MACOSX/._dummy
echo "Это не FinalReport" > test_arch_set01/ignore.txt
cd test_arch_set01
zip -r /app/tests/data/raw/test_data_set01.zip .
cd ..
rm -rf test_arch_set01

# set02
mkdir -p test_arch_set02/__MACOSX test_arch_set02/data/set02 test_arch_set02/maps
cp target_panel_set02_FinalReport.txt test_arch_set02/data/set02/
cp target_panel_set02_SNPMap.txt test_arch_set02/data/set02/
cp target_panel_set02_SampleMap.txt test_arch_set02/maps/
touch test_arch_set02/__MACOSX/._dummy
echo "Это не FinalReport" > test_arch_set02/ignore.txt
cd test_arch_set02
zip -r /app/tests/data/raw/test_data_set02.zip .
cd ..
rm -rf test_arch_set02

# Проверка
unzip -l /app/tests/data/raw/test_data_set01.zip
unzip -l /app/tests/data/raw/test_data_set02.zip


updating: maps/ (stored 0%)
updating: maps/target_panel_set01_SampleMap.txt (deflated 44%)
updating: ignore.txt (stored 0%)
updating: data/ (stored 0%)
updating: data/set01/ (stored 0%)
updating: data/set01/target_panel_set01_FinalReport.txt (deflated 77%)
updating: data/set01/target_panel_set01_SNPMap.txt (deflated 71%)
updating: __MACOSX/ (stored 0%)
updating: __MACOSX/._dummy (stored 0%)
updating: maps/ (stored 0%)
updating: maps/target_panel_set02_SampleMap.txt (deflated 51%)
updating: ignore.txt (stored 0%)
updating: data/ (stored 0%)
updating: data/set02/ (stored 0%)
updating: data/set02/target_panel_set02_FinalReport.txt (deflated 77%)
updating: data/set02/target_panel_set02_SNPMap.txt (deflated 71%)
updating: __MACOSX/ (stored 0%)
updating: __MACOSX/._dummy (stored 0%)
Archive:  /app/tests/data/raw/test_data_set01.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-06-09 09:10   maps/
      263  2026-06-09 09:10   maps/target_panel_set01_Sa

## Фенотипы

In [496]:
sample_columns


Index(['18828', '19830', 'SAMEA5159859', 'SAMEA4644736', 'SAMEA33000418',
       'SAMEA4644767', 'SAMEA32984668', 'SAMEA4780279', 'SAMEA4780314',
       'SAMEA4780255', 'SAMEA4780318', 'SAMEA4780245', 'SAMEA7690246',
       'SAMEA3390141', 'SAMN02843163', 'SAMN02843167', 'SAMN02843162',
       'SAMN02843172', 'SAMN02225739', 'SAMN10940591', 'SAMN10940583',
       'SAMN08612411', 'SAMN08612415', 'SAMN08612382', 'SAMN08612432',
       'SAMN08612412', 'SAMN09519608'],
      dtype='object')

In [497]:
target_panel_all_chr


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,18828,...,SAMN02843172,SAMN02225739,SAMN10940591,SAMN10940583,SAMN08612411,SAMN08612415,SAMN08612382,SAMN08612432,SAMN08612412,SAMN09519608
0,24,112601,24_112601_T_C,T,C,.,PASS,AN=54;AC=21,GT,0/0,...,0/0,0/1,0/1,1/1,0/1,0/0,0/0,0/0,0/1,0/1
1,24,163270,24_163270_C_T,C,T,.,PASS,AN=54;AC=25,GT,0/1,...,1/1,0/1,0/0,1/1,0/1,0/0,0/1,0/0,0/1,0/1
2,24,300821,24_300821_T_C,T,C,.,PASS,AN=54;AC=35,GT,0/1,...,1/1,1/1,0/1,1/1,1/1,0/1,0/1,0/0,1/1,0/1
3,24,335857,24_335857_A_G,A,G,.,PASS,AN=54;AC=13,GT,0/0,...,0/0,1/1,0/0,0/1,0/0,0/0,0/0,0/0,0/0,0/1
4,24,391651,24_391651_A_G,A,G,.,PASS,AN=54;AC=31,GT,1/1,...,0/0,0/1,0/1,1/1,0/1,0/1,0/0,0/0,1/1,0/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2481,25,42167568,25_42167568_T_C,T,C,.,PASS,AN=54;AC=15,GT,0/1,...,0/1,0/1,0/0,0/0,0/1,0/0,1/1,0/1,0/0,0/0
2482,25,42209916,25_42209916_T_C,T,C,.,PASS,AN=54;AC=27,GT,0/0,...,0/0,0/0,1/1,0/0,0/0,1/1,0/0,1/1,0/0,1/1
2483,25,42218897,25_42218897_G_A,G,A,.,PASS,AN=54;AC=17,GT,0/0,...,0/0,0/0,1/1,0/1,0/1,0/0,0/0,0/0,0/0,0/0
2484,25,42244411,25_42244411_C_T,C,T,.,PASS,AN=54;AC=39,GT,0/0,...,0/1,1/1,1/1,1/1,1/1,0/1,1/1,0/1,1/1,0/1


In [498]:
# В GWAS SNP должны быть значимыми
#             "24_112601_T_C",
#             "24_1761048_T_C",
#             "25_42292572_A_G",
#             "25_40905457_A_G",


phenotype = {"ID": [], "Yield": []}
for index, sample in enumerate(sample_columns):
    phenotype["ID"].append(sample)
    phenotype["Yield"].append(0)
    for _, row in target_panel_all_chr.iterrows():
        if row["ID"] in [
            "24_112601_T_C",
            "24_1761048_T_C",
            "25_42292572_A_G",
            "25_40905457_A_G",
        ]:
            gt = row[sample].split("/")
            for i in gt:
                if i == "1":
                    phenotype["Yield"][index] += 1000
                else:
                    phenotype["Yield"][index] += 4
        else:
            phenotype["Yield"][index] += 4


In [499]:
phenotype_df = pd.DataFrame(phenotype)
phenotype_df


,ID,Yield
0,18828,10956
1,19830,10956
2,SAMEA5159859,15936
3,SAMEA4644736,15936
4,SAMEA33000418,13944
5,SAMEA4644767,13944
6,SAMEA32984668,13944
7,SAMEA4780279,13944
8,SAMEA4780314,13944
9,SAMEA4780255,13944


In [500]:
!mkdir tests/data/phenotype


mkdir: cannot create directory ‘tests/data/phenotype’: File exists


In [501]:
phenotype_df.to_csv("tests/data/phenotype/phenotype_all.csv", index=False)
